# Reflective Research Agent: Tool Calling + Iterative Reflection

This notebook builds an autonomous research assistant that can search **arXiv, the web (Tavily), and Wikipedia**, collect evidence, critique its own report, perform additional research when needed, and export the final result.

## Learning objectives
- Define Python functions as LLM-callable tools.
- Use dynamic tool calling and a tool-execution loop.
- Separate planning, evidence gathering, reflection, and report generation.
- Implement bounded iterative reflection.
- Export reports as Markdown, JSON, and HTML.

> **Important:** Wikipedia is useful for background orientation, not as the sole authority for academic claims. Verify important claims against primary sources.


## 1. Install dependencies
Run this cell once in your environment.

In [ ]:
%pip install -q openai tavily-python wikipedia python-dotenv requests

## 2. Configure API keys

Create a `.env` file in the notebook directory:

```text
OPENAI_API_KEY=your_openai_api_key
TAVILY_API_KEY=your_tavily_api_key
```

Do not put real API keys directly in a notebook or commit the `.env` file to GitHub.

In [ ]:
import os
import json
import re
import html
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is missing. Add it to your .env file.")

client = OpenAI()

# Choose a model available to your API account.
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"Client initialized. Model: {MODEL}")
print("Tavily configured:", bool(os.getenv("TAVILY_API_KEY")))

## 3. Load the research tools

The accompanying `utils/research_tools.py` module contains the three search tools and their JSON schemas. Keep the `utils` folder beside this notebook.

In [ ]:
from utils import research_tools

tool_mapping = research_tools.tool_mapping
all_tool_definitions = research_tools.all_tool_definitions

print("Available tools:")
for name in tool_mapping:
    print("-", name)

## 4. Test the tools individually

You can change the queries and options to inspect the returned data before involving the LLM.

In [ ]:
# arXiv: academic papers
arxiv_results = research_tools.arxiv_search_tool(
    query="vision transformers medical image classification",
    max_results=3
)
print(json.dumps(arxiv_results, indent=2, ensure_ascii=False)[:5000])

In [ ]:
# Tavily: general web search
web_results = research_tools.tavily_search_tool(
    query="recent applications of vision transformers in medical imaging",
    max_results=3,
    include_images=False,
    search_depth="advanced"
)
print(json.dumps(web_results, indent=2, ensure_ascii=False)[:5000])

In [ ]:
# Wikipedia: introductory background
wiki_results = research_tools.wikipedia_search_tool(
    query="Vision transformer",
    sentences=4,
    results=2
)
print(json.dumps(wiki_results, indent=2, ensure_ascii=False)[:3000])

## 5. Configure the agent

These settings control the number of tool rounds, allowed tools, response creativity, and the amount of evidence requested.

In [ ]:
AGENT_CONFIG = {
    "model": MODEL,
    "max_tool_rounds": 8,
    "temperature": 0.2,
    "max_results": 5,
    "allowed_tools": [
        "arxiv_search_tool",
        "tavily_search_tool",
        "wikipedia_search_tool"
    ],
    "include_images": False,
    "search_depth": "advanced"
}

def get_tool_definitions(allowed_tools=None):
    """Return only the schemas enabled for this agent run."""
    allowed = set(allowed_tools or tool_mapping.keys())
    return [
        definition for definition in all_tool_definitions
        if definition["function"]["name"] in allowed
    ]

## 6. Research planner

The planner turns a broad question into manageable research dimensions. It does not replace the research agent; it gives the agent a structured starting point.

In [ ]:
def create_research_plan(question: str, model: str = MODEL) -> dict:
    if not question or not question.strip():
        raise ValueError("Research question must not be empty.")

    response = client.chat.completions.create(
        model=model,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a research planning assistant. Return valid JSON with keys: "
                    "research_goal, sub_questions (array), suggested_search_queries (array), "
                    "expected_sections (array), and scope_notes."
                )
            },
            {"role": "user", "content": f"Create a concise research plan for: {question}"}
        ]
    )
    return json.loads(response.choices[0].message.content)

## 7. Tool-calling research agent

The model chooses tools dynamically. The Python program executes each requested function, returns the result to the model, and continues until the model produces a final answer or the round limit is reached.

In [ ]:
RESEARCH_SYSTEM_PROMPT = """You are an evidence-focused research assistant.

Your task is to answer the user's research question using the available tools.
Guidelines:
1. Use arXiv for academic papers, Tavily for broader/current web information,
   and Wikipedia only for introductory background when useful.
2. Choose tools based on the information gap; do not call every tool automatically.
3. You may call multiple tools in one turn when useful.
4. Never invent papers, authors, dates, URLs, statistics, or quotations.
5. Distinguish evidence from interpretation and note conflicting findings.
6. Cite sources with clickable URLs in the report.
7. If the evidence is insufficient, state the limitation clearly.
8. Produce a structured report with an introduction, thematic sections,
   critical discussion, limitations, conclusion, and references.
"""

def run_research_agent(
    question: str,
    plan: dict | None = None,
    model: str = MODEL,
    allowed_tools: list[str] | None = None,
    max_tool_rounds: int = 8,
    temperature: float = 0.2,
    extra_context: str = ""
) -> dict:
    """Run a bounded tool-calling loop and return report, messages, and tool log."""
    if not question or not question.strip():
        raise ValueError("question must not be empty.")

    tool_definitions = get_tool_definitions(allowed_tools)
    plan_text = json.dumps(plan, ensure_ascii=False, indent=2) if plan else "No separate plan."
    user_content = (
        f"Research question:\n{question}\n\n"
        f"Research plan:\n{plan_text}\n\n"
        f"Additional context / research gaps:\n{extra_context or 'None'}"
    )
    messages = [
        {"role": "system", "content": RESEARCH_SYSTEM_PROMPT},
        {"role": "user", "content": user_content}
    ]
    tool_log = []
    final_report = ""

    for round_number in range(1, max_tool_rounds + 1):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tool_definitions,
            tool_choice="auto" if tool_definitions else "none",
            temperature=temperature
        )
        assistant_message = response.choices[0].message
        messages.append(assistant_message.model_dump(exclude_none=True))

        if not assistant_message.tool_calls:
            final_report = assistant_message.content or ""
            break

        for tool_call in assistant_message.tool_calls:
            tool_name = tool_call.function.name
            try:
                arguments = json.loads(tool_call.function.arguments or "{}")
                if tool_name not in (allowed_tools or tool_mapping.keys()):
                    raise ValueError(f"Tool '{tool_name}' is not enabled.")
                tool_function = tool_mapping[tool_name]
                tool_result = tool_function(**arguments)
                tool_log.append({
                    "round": round_number,
                    "tool": tool_name,
                    "arguments": arguments,
                    "result": tool_result
                })
                tool_content = json.dumps(tool_result, ensure_ascii=False, default=str)
            except Exception as exc:
                tool_content = json.dumps({"error": str(exc)}, ensure_ascii=False)
                tool_log.append({
                    "round": round_number,
                    "tool": tool_name,
                    "error": str(exc)
                })

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": tool_content
            })

    if not final_report:
        # One final response without tools prevents an unfinished tool-call cycle.
        messages.append({
            "role": "user",
            "content": "Using the evidence gathered so far, provide the best complete report possible. State any evidence gaps."
        })
        final_response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature
        )
        final_report = final_response.choices[0].message.content or ""

    return {
        "question": question,
        "report": final_report,
        "messages": messages,
        "tool_log": tool_log,
        "tool_calls_count": len(tool_log)
    }

## 8. Reflection / critic

The critic checks the report for missing evidence, unsupported claims, source diversity, balance, and completeness. It returns machine-readable JSON so the workflow can decide whether another research pass is needed.

In [ ]:
REFLECTION_SYSTEM_PROMPT = """You are a strict but constructive academic reviewer.
Evaluate the report against the research question and evidence.
Return valid JSON with exactly these keys:
{
  "approved": boolean,
  "quality_score": integer from 1 to 10,
  "strengths": [string],
  "limitations": [string],
  "unsupported_or_unclear_claims": [string],
  "missing_topics": [string],
  "additional_search_queries": [string],
  "revision_instructions": [string]
}
Be conservative: approve only when the report is sufficiently complete and evidence-based.
Do not ask for more research merely to make the report longer.
"""

def reflect_on_report(
    question: str,
    report: str,
    tool_log: list[dict] | None = None,
    model: str = MODEL
) -> dict:
    evidence_summary = []
    for item in (tool_log or []):
        evidence_summary.append({
            "tool": item.get("tool"),
            "arguments": item.get("arguments"),
            "error": item.get("error")
        })

    response = client.chat.completions.create(
        model=model,
        temperature=0.1,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": REFLECTION_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Research question:\n{question}\n\n"
                    f"Tool-use record:\n{json.dumps(evidence_summary, ensure_ascii=False)}\n\n"
                    f"Report to review:\n{report}"
                )
            }
        ]
    )
    return json.loads(response.choices[0].message.content)

## 9. Iterative reflective research

This is the main agent workflow:
1. Create a research plan.
2. Conduct tool-assisted research.
3. Reflect on the report.
4. If the critic identifies important gaps, search those gaps in another pass.
5. Revise the report using the additional evidence.
6. Stop when approved or when the reflection limit is reached.

The iteration limit prevents endless self-correction and unexpected API costs.

In [ ]:
def run_reflective_research(
    question: str,
    model: str = MODEL,
    max_reflections: int = 2,
    max_tool_rounds: int = 8,
    allowed_tools: list[str] | None = None,
    temperature: float = 0.2,
    verbose: bool = True
) -> dict:
    plan = create_research_plan(question, model=model)

    research = run_research_agent(
        question=question,
        plan=plan,
        model=model,
        allowed_tools=allowed_tools,
        max_tool_rounds=max_tool_rounds,
        temperature=temperature
    )

    reflections = []
    combined_log = list(research["tool_log"])
    current_report = research["report"]

    for iteration in range(1, max_reflections + 1):
        critique = reflect_on_report(
            question=question,
            report=current_report,
            tool_log=combined_log,
            model=model
        )
        reflections.append(critique)

        if verbose:
            print(f"Reflection {iteration}: score={critique.get('quality_score')}/10; "
                  f"approved={critique.get('approved')}")

        if critique.get("approved", False):
            break

        queries = critique.get("additional_search_queries", [])
        instructions = critique.get("revision_instructions", [])
        missing = critique.get("missing_topics", [])

        if not queries and not missing:
            break

        gap_context = (
            "Please investigate these research gaps and improve the existing report.\n"
            f"Additional search queries: {json.dumps(queries, ensure_ascii=False)}\n"
            f"Missing topics: {json.dumps(missing, ensure_ascii=False)}\n"
            f"Revision instructions: {json.dumps(instructions, ensure_ascii=False)}\n\n"
            f"Previous report:\n{current_report}"
        )

        follow_up = run_research_agent(
            question=question,
            plan=plan,
            model=model,
            allowed_tools=allowed_tools,
            max_tool_rounds=max_tool_rounds,
            temperature=temperature,
            extra_context=gap_context
        )
        combined_log.extend(follow_up["tool_log"])
        current_report = follow_up["report"]

    # Final editorial pass: use the collected critique, but do not invent evidence.
    final_response = client.chat.completions.create(
        model=model,
        temperature=0.2,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an academic editor. Improve the report using the reviewer feedback. "
                    "Preserve valid citations and URLs. Do not invent evidence or references. "
                    "Explicitly acknowledge unresolved limitations."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    f"Latest report:\n{current_report}\n\n"
                    f"Reviewer feedback:\n{json.dumps(reflections, ensure_ascii=False)}"
                )
            }
        ]
    )
    final_report = final_response.choices[0].message.content or current_report

    return {
        "question": question,
        "plan": plan,
        "initial_report": research["report"],
        "final_report": final_report,
        "reflections": reflections,
        "tool_log": combined_log,
        "tool_calls_count": len(combined_log),
        "created_at": datetime.now(timezone.utc).isoformat()
    }

## 10. Export the report

The following helpers save the final report and research metadata as Markdown, JSON, and HTML. HTML conversion uses the LLM for presentation; Markdown and JSON preserve the report text directly.

In [ ]:
def export_report(result: dict, output_dir: str = "research_output", filename: str = "research_report"):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    report = result.get("final_report", "")
    md_path = output_path / f"{filename}.md"
    json_path = output_path / f"{filename}.json"
    html_path = output_path / f"{filename}.html"

    md_path.write_text(report, encoding="utf-8")
    json_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8"
    )

    safe_report = html.escape(report).replace("\n", "<br>\n")
    html_document = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Reflective Research Report</title>
<style>
body {{ max-width: 900px; margin: 40px auto; padding: 0 24px;
       font: 16px/1.7 system-ui, sans-serif; color: #202124; }}
main {{ background: #fff; }} h1,h2,h3 {{ line-height: 1.3; }}
pre {{ white-space: pre-wrap; }}
</style>
</head>
<body><main><h1>Research Report</h1><pre>{safe_report}</pre></main></body>
</html>"""
    html_path.write_text(html_document, encoding="utf-8")

    return {"markdown": str(md_path), "json": str(json_path), "html": str(html_path)}

## 11. Run the complete agent

Change the research question and options as needed. For a first test, keep `max_reflections` low to control cost and execution time.

In [ ]:
research_question = (
    "Compare convolutional neural networks and vision transformers "
    "for medical image classification, focusing on data requirements, "
    "generalization, interpretability, and recent research findings."
)

result = run_reflective_research(
    question=research_question,
    model=MODEL,
    max_reflections=2,
    max_tool_rounds=8,
    allowed_tools=[
        "arxiv_search_tool",
        "tavily_search_tool",
        "wikipedia_search_tool"
    ],
    temperature=0.2,
    verbose=True
)

print("\n=== FINAL RESEARCH REPORT ===\n")
print(result["final_report"])

print("\n=== TOOL USAGE ===")
print("Total tool calls:", result["tool_calls_count"])
for item in result["tool_log"]:
    print(item.get("tool"), item.get("arguments", item.get("error", "")))

In [ ]:
# Export all output formats
export_paths = export_report(result)
print(export_paths)

# Display the HTML file in the notebook
display(HTML(Path(export_paths["html"]).read_text(encoding="utf-8")))

## 12. Experiment with different configurations

Try these variations:
- Disable Wikipedia and compare the evidence.
- Use only arXiv for a literature-focused review.
- Increase Tavily search depth to `advanced`.
- Set `max_reflections=0` to compare against a non-reflective baseline.
- Compare tool-call counts and reflection scores across runs.
- Ask the agent to investigate a topic with conflicting findings.

Example tool subsets:

In [ ]:
# Academic-only agent
academic_tools = ["arxiv_search_tool"]

# Web research agent
web_tools = ["tavily_search_tool"]

# Multi-source research agent
all_research_tools = [
    "arxiv_search_tool",
    "tavily_search_tool",
    "wikipedia_search_tool"
]

print("Academic-only:", academic_tools)
print("Web-only:", web_tools)
print("Multi-source:", all_research_tools)